# Fine-Tuning the RAG Embedding Model

This notebook trains our baseline sentence transformer (`all-MiniLM-L6-v2`) to better understand the relationship between developer questions and raw code.

We use **Multiple Negatives Ranking Loss (MNRL)**. With MNRL, we provide pairs of `(Question, Correct Code)`. The loss function automatically treats all other code snippets in the batch as *negative* samples, heavily penalizing the model if it ranks them higher than the correct code.

In [ ]:
import os
import ast
import json
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

print("Imports successful!")

## 1. Load the Generated Dataset
First, we load the synthetic data we generated via the Groq API.

In [ ]:
dataset_path = 'training_data.jsonl'
data = []

with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

df = pd.DataFrame(data)
print(f"Loaded {len(df)} training pairs.")
df.head(2)

## 2. Prepare Data for Sentence Transformers
MNRL requires data wrapped in `InputExample` objects, containing a list of `[Anchor, Positive]`.

In [ ]:
train_examples = []
for index, row in df.iterrows():
    # Anchor = User Question, Positive = Code Chunk
    train_examples.append(InputExample(texts=[row['question'], row['code']]))

# The batch size is crucial for MNRL. Larger batches = more "negatives" to learn from.
# 16 is a good starting point for a small memory footprint.
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
print(f"Created DataLoader with {len(train_dataloader)} batches.")

## 3. Initialize Model & Loss

In [ ]:
model_name = 'all-MiniLM-L6-v2'
print(f"Loading base model: {model_name}")
model = SentenceTransformer(model_name)

# Define the MNRL objective
train_loss = losses.MultipleNegativesRankingLoss(model=model)

## 4. Train the Model

In [ ]:
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of train data for warm-up

output_dir = '../models/finetuned-explainer-model'

print("Starting training...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=output_dir,
    show_progress_bar=True
)

print(f"Training complete! Model successfully saved to {output_dir}")